In [0]:
-------------------------------
--Customer Dimension (SCD2)
---------------------------------
CREATE TABLE IF NOT EXISTS gold.dim_customer (
    customer_key BIGINT GENERATED ALWAYS AS IDENTITY,
    customer_id INT,
    customer_number STRING,
    first_name STRING,
    last_name STRING,
    country STRING,
    marital_status STRING,
    gender STRING,
    birthdate DATE,
    create_date DATE,
    effective_from TIMESTAMP,
    effective_to TIMESTAMP,
    is_current BOOLEAN,
    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING DELTA;

---------------------------------
--Product Dimension (SCD2)
---------------------------------

CREATE TABLE IF NOT EXISTS gold.dim_product (
    product_key BIGINT GENERATED ALWAYS AS IDENTITY,
    product_id INT,
    product_number STRING,
    product_name STRING,
    category STRING,
    subcategory STRING,
    maintenance STRING,
    cost DOUBLE,
    product_line STRING,
    start_date DATE,
    end_date DATE,
    effective_from TIMESTAMP,
    effective_to TIMESTAMP,
    is_current BOOLEAN,
    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING DELTA;

---------------------------------
--Fact Table
---------------------------------
CREATE TABLE IF NOT EXISTS gold.fact_sale (
    order_number STRING,
    product_key BIGINT,
    customer_key BIGINT,
    order_date DATE,
    shipping_date DATE,
    due_date DATE,
    sales_amount DOUBLE,
    quantity INT,
    price DOUBLE,
    created_at TIMESTAMP
  )
USING DELTA;






In [0]:

----------------------------------
--Expire Old Records-Dim Customer
----------------------------------
MERGE INTO gold.dim_customer tgt
USING (
    SELECT
        ci.cst_id AS customer_id,
        ci.cst_key AS customer_number,
        ci.cst_firstname AS first_name,
        ci.cst_lastname AS last_name,
        la.cntry AS country,
        ci.cst_marital_status AS marital_status,

        CASE 
            WHEN ci.cst_gndr != 'n/a' THEN ci.cst_gndr
            ELSE COALESCE(ca.gen, 'n/a')
        END AS gender,

        ca.bdate AS birthdate,
        ci.cst_create_date AS create_date

    FROM silver.crm_cust_info ci
    LEFT JOIN silver.erp_cust_az12 ca
        ON ci.cst_id = ca.cid
    LEFT JOIN silver.erp_loc_a101 la
        ON ci.cst_id = la.cid
) src
ON tgt.customer_id = src.customer_id
AND tgt.is_current = true

WHEN MATCHED AND (
    tgt.first_name <> src.first_name OR
    tgt.last_name <> src.last_name OR
    tgt.country <> src.country OR
    tgt.marital_status <> src.marital_status OR
    tgt.gender <> src.gender
)
THEN UPDATE SET
    tgt.effective_to = CURRENT_TIMESTAMP(),
    tgt.is_current = false,
    tgt.updated_at = CURRENT_TIMESTAMP();


    ----------------------------------
--Insert New Records-Dim Customer
--------------------------------------
INSERT INTO gold.dim_customer (
    customer_id,
    customer_number,
    first_name,
    last_name,
    country,
    marital_status,
    gender,
    birthdate,
    create_date,
    effective_from,
    effective_to,
    is_current,
    created_at,
    updated_at
       
)
SELECT
    src.customer_id,
    src.customer_number,
    src.first_name,
    src.last_name,
    src.country,
    src.marital_status,
    src.gender,
    src.birthdate,
    src.create_date,
    CURRENT_TIMESTAMP(),
    NULL,
    true,
    CURRENT_TIMESTAMP(), -- created_at
    CURRENT_TIMESTAMP()  -- updated_at
   
FROM (
    SELECT
        ci.cst_id AS customer_id,
        ci.cst_key AS customer_number,
        ci.cst_firstname AS first_name,
        ci.cst_lastname AS last_name,
        la.cntry AS country,
        ci.cst_marital_status AS marital_status,

        CASE 
            WHEN ci.cst_gndr != 'n/a' THEN ci.cst_gndr
            ELSE COALESCE(ca.gen, 'n/a')
        END AS gender,

        ca.bdate AS birthdate,
        ci.cst_create_date AS create_date

    FROM silver.crm_cust_info ci
    LEFT JOIN silver.erp_cust_az12 ca
        ON ci.cst_id = ca.cid
    LEFT JOIN silver.erp_loc_a101 la
        ON ci.cst_id = la.cid
) src
LEFT JOIN gold.dim_customer tgt
    ON src.customer_id = tgt.customer_id
    AND tgt.is_current = true
WHERE tgt.customer_id IS NULL;


In [0]:
----------------------------------
--Expire Old Records-Dim Product
----------------------------------
--truncate table gold.dim_product
MERGE INTO gold.dim_product tgt
USING (
    SELECT
        pn.prd_id AS product_id,
        pn.prd_key AS product_number,
        pn.prd_nm AS product_name,
        pc.cat AS category,
        pc.subcat AS subcategory,
        pc.maintenance AS maintenance,
        pn.prd_cost AS cost,
        pn.prd_line AS product_line,
        pn.prd_start_dt AS start_date,
        pn.prd_end_dt AS end_date
    FROM silver.crm_prd_info pn
    LEFT JOIN silver.erp_px_cat_g1v2 pc
        ON pn.cat_id = pc.id
   -- WHERE pn.prd_end_dt IS NULL
) src
ON tgt.product_id = src.product_id
AND tgt.is_current = true

WHEN MATCHED AND (
    tgt.product_number <> src.product_number OR
    tgt.product_name <> src.product_name OR
    tgt.cost <> src.cost OR
    tgt.category <> src.category
)
THEN UPDATE SET
    tgt.product_number = src.product_number ,
    tgt.product_name = src.product_name ,
    tgt.cost = src.cost ,
    tgt.category = src.category,
    tgt.subcategory = src.subcategory,
    tgt.maintenance = src.maintenance,
    tgt.product_line = src.product_line,
    tgt.effective_from = CURRENT_TIMESTAMP(),
    tgt.effective_to = CURRENT_TIMESTAMP(),
    tgt.is_current = false,
    tgt.updated_at = CURRENT_TIMESTAMP()
    
   -- WHEN NOT MATCHED THEN INSERT *
    ;

       
 ----------------------------------
--Insert New Records-Dim Product
--------------------------------------
INSERT INTO gold.dim_product (
    product_id,
    product_number,
    product_name,
    category,
    subcategory,
    maintenance,
    cost,
    product_line,
    start_date,
    end_date,
    effective_from,
    effective_to,
    is_current,
    created_at,
    updated_at
)
SELECT
    src.product_id,
    src.product_number,
    src.product_name,
    src.category,
    src.subcategory,
    src.maintenance,
    src.cost,
    src.product_line,
    src.start_date,
    src.end_date,
    CURRENT_TIMESTAMP(),
    NULL,
    true,
    CURRENT_TIMESTAMP(),
    CURRENT_TIMESTAMP()
FROM (
    SELECT
        pn.prd_id AS product_id,
        pn.prd_key AS product_number,
        pn.prd_nm AS product_name,
        pc.cat AS category,
        pc.subcat AS subcategory,
        pc.maintenance AS maintenance,
        pn.prd_cost AS cost,
        pn.prd_line AS product_line,
        pn.prd_start_dt AS start_date,
        pn.prd_end_dt AS end_date
    FROM silver.crm_prd_info pn
    LEFT JOIN silver.erp_px_cat_g1v2 pc
        ON pn.cat_id = pc.id
    --WHERE pn.prd_end_dt IS NULL
) src
LEFT JOIN gold.dim_product tgt
    ON src.product_id = tgt.product_id
    AND tgt.is_current = true
WHERE tgt.product_id IS NULL;